# Mat & Comp – RN 1: El perceptrón
## La primera neurona artificial

**Curso:** Redes neuronales (Sección 4) · **Duración sugerida:** 2 sesiones de 50 minutos · **Requisito:** ML 7

### Objetivos de aprendizaje
Al terminar esta lección podrás:
1. Describir una **neurona artificial**: entradas, **pesos**, **sesgo** y **función de activación**.
2. Calcular a mano la salida de un perceptrón y diseñar pesos para las compuertas lógicas **Y** y **O**.
3. Interpretar un perceptrón como una **recta** que divide el plano en dos.
4. Programar desde cero la **regla de aprendizaje del perceptrón** y verla funcionar.
5. Entender por qué un perceptrón **no puede** aprender el **O exclusivo (XOR)**, y por qué eso lleva a las redes de varias capas.

## 🔍 ¿Voy al partido?

El sábado juega tu equipo y debes decidir si vas al estadio. Piensas en tres cosas:

| Pregunta | Sí = 1, No = 0 | ¿Cuánto me importa? |
|---|---|---|
| ¿Van mis amigos? | $x_1$ | mucho: **+3** |
| ¿Hace buen clima? | $x_2$ | algo: **+1** |
| ¿Tengo un examen el lunes? | $x_3$ | en contra: **−2** |

Sumas los puntos de lo que se cumple y vas solo si el total pasa de **2** (te da pereza salir).

✏️ *Si van tus amigos, llueve y tienes examen, ¿vas? ¿Y si no van tus amigos pero hace sol y no tienes examen?*

**Mis respuestas:**

Acabas de hacer exactamente lo que hace una **neurona artificial**. En 1958, el psicólogo Frank Rosenblatt construyó una máquina basada en esta idea y la llamó **perceptrón**. Hoy es la pieza básica de todas las redes neuronales, incluidas las que reconocen caras o conversan contigo.

## 1. Anatomía de una neurona artificial

Una neurona del cerebro recibe señales de otras por sus dendritas; si la señal total es suficientemente fuerte, «dispara» un impulso. La neurona artificial copia esa idea:

1. Recibe **entradas** $x_1, x_2, \dots$
2. Multiplica cada una por su **peso** $w_1, w_2, \dots$ (qué tan importante es).
3. Suma todo y le agrega un **sesgo** $b$ (qué tan fácil o difícil es que dispare).
4. Aplica una **función de activación**: si la suma es positiva, la salida es 1; si no, 0.

$$z = w_1 x_1 + w_2 x_2 + w_3 x_3 + b \qquad \text{salida} = \begin{cases} 1 & \text{si } z > 0 \\ 0 & \text{si } z \le 0 \end{cases}$$

En el partido: los pesos son 3, 1 y −2, y «pasar de 2» equivale a un sesgo $b = -2$.

Con numpy, la suma $w_1 x_1 + w_2 x_2 + w_3 x_3$ es el **producto punto** `x @ w`: multiplica elemento por elemento y suma.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def escalon(z):
    return np.where(z > 0, 1, 0)

def neurona(x, w, b):
    z = x @ w + b
    return escalon(z)

w = np.array([3, 1, -2])     # amigos, clima, examen
b = -2

print(np.array([1, 0, 1]) @ w, "→ z =", np.array([1, 0, 1]) @ w + b)
print("Amigos sí, lluvia, examen:", neurona(np.array([1, 0, 1]), w, b))
print("Sin amigos, sol, sin examen:", neurona(np.array([0, 1, 0]), w, b))

Probemos las 8 situaciones posibles de una vez. Si `X` tiene una situación por fila, `X @ w` calcula las 8 sumas a la vez:

In [ ]:
situaciones = np.array([[a, c, e] for a in [0, 1] for c in [0, 1] for e in [0, 1]])
decisiones = neurona(situaciones, w, b)

print("amigos clima examen → ¿voy?")
for s, d in zip(situaciones, decisiones):
    print(f"  {s[0]}      {s[1]}      {s[2]}    →  {'sí' if d else 'no'}")

✏️ *¿Qué tendrías que cambiar para que el examen te detuviera **siempre**, aunque vayan tus amigos y haga sol?*

## 2. Pensar con compuertas lógicas

Las compuertas lógicas son las piezas de los computadores. Con dos entradas que valen 0 o 1:

| $x_1$ | $x_2$ | Y (AND) | O (OR) |
|---|---|---|---|
| 0 | 0 | 0 | 0 |
| 0 | 1 | 0 | 1 |
| 1 | 0 | 0 | 1 |
| 1 | 1 | 1 | 1 |

¿Puede una neurona hacer de compuerta? Para **Y**, con pesos 1 y 1, la suma vale 0, 1 o 2. Queremos que dispare solo con 2, así que el sesgo debe «exigir» más de 1: por ejemplo $b = -1.5$.

In [ ]:
entradas = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])

print("Y :", neurona(entradas, np.array([1, 1]), -1.5))
print("O :", neurona(entradas, np.array([1, 1]), -0.5))

✏️ *¿Por qué el O funciona con $b = -0.5$? ¿Funcionaría también con $b = -0.9$? ¿Y con $b = -1.2$?*

## 3. Un perceptrón es una recta

Con dos entradas, el perceptrón dispara cuando $w_1 x_1 + w_2 x_2 + b > 0$. La frontera $w_1 x_1 + w_2 x_2 + b = 0$ es una **recta**: de un lado la salida es 1, del otro es 0. Despejando $x_2$:

$$x_2 = -\frac{w_1}{w_2} x_1 - \frac{b}{w_2}$$

Dibujemos la recta de la compuerta Y y la de la O:

In [ ]:
def dibujar_frontera(eje, w, b, x_min=-0.5, x_max=1.5, color="crimson"):
    xs = np.linspace(x_min, x_max, 100)
    eje.plot(xs, -(w[0] * xs + b) / w[1], color=color)

fig, ejes = plt.subplots(1, 2, figsize=(10, 4.5))
for eje, (nombre, b) in zip(ejes, [("Y", -1.5), ("O", -0.5)]):
    w = np.array([1, 1])
    salida = neurona(entradas, w, b)
    eje.scatter(entradas[:, 0], entradas[:, 1], c=np.where(salida == 1, "seagreen", "crimson"), s=200, zorder=3)
    dibujar_frontera(eje, w, b)
    eje.set_xlim(-0.5, 1.5); eje.set_ylim(-0.5, 1.5)
    eje.set_title(f"Compuerta {nombre}: x₁ + x₂ {b:+} = 0")
    eje.set_xlabel("x₁"); eje.set_ylabel("x₂")
    eje.set_aspect("equal"); eje.grid(alpha=0.3)
plt.show()

Los puntos verdes (salida 1) quedan a un lado de la recta y los rojos (salida 0) al otro. Los **pesos** deciden la **inclinación** de la recta y el **sesgo** la **desplaza**.

✏️ *En la celda de abajo, cambia los pesos y el sesgo y observa cómo se mueve la recta. ¿Qué pasa si cambias el signo de todos?*

In [ ]:
w = np.array([1.0, 2.0])     # cambia estos números
b = -1.0

fig, eje = plt.subplots(figsize=(5, 5))
xs1, xs2 = np.meshgrid(np.linspace(-1, 2, 200), np.linspace(-1, 2, 200))
Z = escalon(w[0] * xs1 + w[1] * xs2 + b)
eje.contourf(xs1, xs2, Z, levels=[-0.5, 0.5, 1.5], colors=["#f8d0d0", "#cdebd6"])
dibujar_frontera(eje, w, b, -1, 2)
eje.set_xlim(-1, 2); eje.set_ylim(-1, 2)
eje.set_title(f"w = {w}, b = {b}   (verde: salida 1)")
eje.set_aspect("equal")
plt.show()

## 4. ¿Cómo aprende? La regla del perceptrón

Hasta ahora elegimos los pesos nosotros. La gran idea de Rosenblatt fue que la neurona los **aprenda sola** a partir de ejemplos. La regla es muy simple. Para cada ejemplo $(x, y)$:

1. Calcular la predicción $\hat{y}$.
2. Calcular el error: $\text{error} = y - \hat{y}$. Vale **0** si acertó, **+1** si debía decir 1 y dijo 0, y **−1** si debía decir 0 y dijo 1.
3. Corregir:
$$w \leftarrow w + \eta \cdot \text{error} \cdot x \qquad\qquad b \leftarrow b + \eta \cdot \text{error}$$

donde $\eta$ (la letra griega «eta») es la **tasa de aprendizaje**, como en ML 3.

**¿Por qué funciona?** Si la neurona dijo 0 y debía decir 1, la suma $z$ era muy pequeña. Sumarle $x$ a los pesos hace que la próxima vez $z$ crezca justo para ese ejemplo. Si dijo 1 y debía decir 0, restar hace lo contrario. Si acertó, no se cambia nada.

Veámosla aprender la compuerta **Y** desde cero, mostrando cada paso:

In [ ]:
y_and = np.array([0, 0, 0, 1])
w = np.zeros(2)
b = 0.0
eta = 0.5

for vuelta in range(1, 11):
    print(f"--- Vuelta {vuelta} ---")
    errores = 0
    for x, y in zip(entradas, y_and):
        pred = neurona(x, w, b)
        error = y - pred
        if error != 0:
            w = w + eta * error * x
            b = b + eta * error
            errores += 1
        print(f"x = {x}  y = {y}  pred = {pred}  error = {error:+d}  →  w = {w}, b = {b:+.1f}")
    if errores == 0:
        print("¡Ninguna equivocación en toda la vuelta! Aprendió.")
        break

print("Resultado:", neurona(entradas, w, b))

Cada pasada completa por todos los ejemplos se llama una **época** (aquí, cada «vuelta»). El perceptrón aprendió la compuerta Y en 6 épocas, sin que nadie le dijera los pesos. Fíjate que a veces una corrección «daña» un ejemplo que ya estaba bien; aun así, poco a poco llega.

## 5. Aprender a separar frutas

Ahora un problema con datos. En una finca clasifican mandarinas en **maduras** (1) y **verdes** (0) según dos medidas: el **color** (de 0 = verde a 10 = naranja) y la **dureza** (de 0 = blanda a 10 = dura).

In [ ]:
rng = np.random.default_rng(0)
n = 20
maduras = np.column_stack([rng.normal(7, 1.0, n), rng.normal(3.5, 1.0, n)]).round(1)
verdes  = np.column_stack([rng.normal(3.5, 1.0, n), rng.normal(6.5, 1.0, n)]).round(1)
X = np.vstack([maduras, verdes])
y = np.array([1] * n + [0] * n)

plt.scatter(maduras[:, 0], maduras[:, 1], color="orange", label="madura (1)")
plt.scatter(verdes[:, 0], verdes[:, 1], color="green", label="verde (0)")
plt.xlabel("color"); plt.ylabel("dureza")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

Escribimos la regla como una función. En cada época recorremos los ejemplos **en desorden** (como en el descenso de gradiente estocástico del reto de ML 3) y guardamos los pesos para ver cómo se mueve la recta:

In [ ]:
def entrenar_perceptron(X, y, eta=0.1, epocas=30, semilla=0):
    rng = np.random.default_rng(semilla)
    w = np.zeros(X.shape[1])
    b = 0.0
    historia = []                         # (w, b, errores) al final de cada época
    for epoca in range(epocas):
        errores = 0
        for i in rng.permutation(len(X)):
            error = y[i] - neurona(X[i], w, b)
            if error != 0:
                w = w + eta * error * X[i]
                b = b + eta * error
                errores += 1
        historia.append((w.copy(), b, errores))
        if errores == 0:                  # una época perfecta: terminamos
            break
    return w, b, historia

w, b, historia = entrenar_perceptron(X, y)
print("Errores en cada época:", [h[2] for h in historia])
print("Pesos finales:", w.round(2), " sesgo:", round(b, 2))
print(f"Exactitud: {np.mean(neurona(X, w, b) == y):.0%}")

In [ ]:
mostrar = range(len(historia))            # una gráfica por época
fig, ejes = plt.subplots(1, len(historia), figsize=(5 * len(historia), 4.5), sharey=True)
for eje, e in zip(ejes, mostrar):
    w_e, b_e, err = historia[e]
    eje.scatter(maduras[:, 0], maduras[:, 1], color="orange", s=20)
    eje.scatter(verdes[:, 0], verdes[:, 1], color="green", s=20)
    dibujar_frontera(eje, w_e, b_e, 0, 10)
    eje.set_xlim(0, 10); eje.set_ylim(0, 10)
    eje.set_title(f"después de la época {e + 1}: {err} errores")
    eje.set_xlabel("color")
ejes[0].set_ylabel("dureza")
plt.show()

Con solo tres épocas, la recta se acomoda hasta separar las dos clases sin errores. Un teorema demostrado en 1962 garantiza que, si **existe** una recta que separa las clases (se dice que los datos son **linealmente separables**), el perceptrón la encuentra en un número finito de pasos.

## 6. ⚠️ El problema del O exclusivo (XOR)

El **O exclusivo** (XOR) dice 1 cuando las entradas son **distintas**: «de postre como helado **o** torta, pero no las dos cosas».

| $x_1$ | $x_2$ | XOR |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

In [ ]:
y_xor = np.array([0, 1, 1, 0])
w, b, historia = entrenar_perceptron(entradas, y_xor, eta=0.5, epocas=20)
print("Errores en cada época:", [h[2] for h in historia])
print("Predicciones finales:", neurona(entradas, w, b), " (queríamos", y_xor, ")")

plt.figure(figsize=(4.5, 4.5))
plt.scatter(entradas[:, 0], entradas[:, 1], c=np.where(y_xor == 1, "seagreen", "crimson"), s=250)
plt.xlim(-0.5, 1.5); plt.ylim(-0.5, 1.5)
plt.title("XOR: verde = 1, rojo = 0")
plt.gca().set_aspect("equal"); plt.grid(alpha=0.3)
plt.show()

Los errores **nunca** llegan a cero. Mira el dibujo: los dos puntos verdes están en una diagonal y los dos rojos en la otra. ✏️ *Intenta trazar con el dedo una sola recta que deje los verdes de un lado y los rojos del otro.* Es imposible: el XOR **no es linealmente separable**.

En 1969, Marvin Minsky y Seymour Papert publicaron este y otros límites del perceptrón en un libro muy influyente. El entusiasmo por las redes neuronales cayó durante años (se habla de un «invierno» de la IA). La salida llegó al conectar **varias neuronas en capas**: una red con una capa intermedia sí puede aprender el XOR. Eso es lo que construiremos en RN 2.

## 7. Datos reales: ¿aprobó?

Probemos el perceptrón con los estudiantes, usando horas de estudio y de sueño **normalizadas** (como en ML 4).

In [ ]:
import pandas as pd

datos = pd.read_csv("https://raw.githubusercontent.com/emersonjleon/compumath/main/lecciones/seccion3_ml/datos/estudiantes.csv").dropna()
Xe = datos[["horas_estudio", "horas_sueno"]].to_numpy()
Xe = (Xe - Xe.mean(axis=0)) / Xe.std(axis=0)
ye = (datos["nota_matematicas"] >= 3.0).astype(int).to_numpy()   # 1 = aprobó

w, b, historia = entrenar_perceptron(Xe, ye, eta=0.1, epocas=40)
errores = [h[2] for h in historia]
print("Errores por época:", errores)
print(f"Exactitud final: {np.mean(neurona(Xe, w, b) == ye):.0%}")

plt.plot(range(1, len(errores) + 1), errores, "o-")
plt.xlabel("época"); plt.ylabel("errores en la época")
plt.title("El perceptrón con los estudiantes")
plt.grid(alpha=0.3)
plt.show()

Los datos reales **no** son perfectamente separables (hay estudiantes que estudiaron mucho y no aprobaron, y al revés). Entonces la regla nunca deja de corregir: la recta «tiembla» de una época a otra y los errores no llegan a cero. Aun así, la recta final acierta la gran mayoría. Esa inestabilidad es otra razón para buscar algo mejor que el escalón, como veremos en RN 2 y RN 3.

## 8. Con scikit-learn

scikit-learn tiene la clase `Perceptron`, que usa la misma regla:

In [ ]:
from sklearn.linear_model import Perceptron

modelo = Perceptron(random_state=0)
modelo.fit(X, y)                                  # las frutas

print("Pesos:", modelo.coef_, " sesgo:", modelo.intercept_)
print(f"Exactitud con las frutas: {modelo.score(X, y):.0%}")
print("Una fruta de color 6 y dureza 4:", modelo.predict([[6, 4]]))

Los pesos no son los mismos que los nuestros (hay muchas rectas que separan los datos y depende del orden en que vea los ejemplos), pero también separa perfectamente las frutas.

## 9. Errores comunes

**Error 1: olvidar normalizar.** Con los estudiantes sin normalizar, las horas de estudio (hasta 14) y el sesgo (que cambia de a 0.1) quedan en escalas muy distintas y el aprendizaje es mucho peor. ¡Pruébalo en la sección 7!

**Error 2: esperar que siempre converja.** Si los datos no son linealmente separables, el perceptrón nunca termina de corregir. Hay que ponerle un número máximo de épocas.

**Error 3: confundir pesos y entradas.** Las entradas son los datos (no se cambian); los pesos y el sesgo son lo que la neurona aprende.

**Error 4: pasar un solo ejemplo como lista simple** a scikit-learn (como en ML 4):

In [ ]:
modelo.predict([6, 4])

## 🤔 ¿Qué pasa si…?

Usa la celda de abajo. Antes de ejecutar, **predice** el resultado.

1. …en el partido, cambias el sesgo a $b = 0$? ¿Y a $b = -4$? ¿Cuántas de las 8 situaciones dicen «sí» en cada caso?
2. …entrenas las frutas con `eta=0.01` y con `eta=10`? Compara los errores por época y la exactitud final. (Pista: empezando con pesos en cero, ¿qué hace η con los pesos?)
3. …entrenas la compuerta O con `entrenar_perceptron`? ¿En cuántas épocas aprende?
4. …multiplicas todos los pesos **y** el sesgo de la compuerta Y por 10? ¿Cambian las salidas?
5. …entrenas a los estudiantes **sin** normalizar? ¿Cuál es la exactitud final?

In [ ]:
# Prueba aquí tus ideas

## ✍️ Ejercicios

**Ejercicio 1 (fácil). La compuerta NO.** La compuerta NO tiene una sola entrada y la invierte: 0 → 1 y 1 → 0. Encuentra un peso y un sesgo que la hagan, y compruébalo con `neurona`.

**Ejercicio 2 (fácil). La compuerta NO-Y (NAND).** NAND da 0 solo cuando las dos entradas son 1. Encuentra pesos y sesgo a mano y dibuja su recta. (Pista: compárala con la compuerta Y.)

**Ejercicio 3 (medio). Paso a paso.** Copia la celda de la sección 4 y úsala para que el perceptrón aprenda la compuerta **O** con `eta = 1`, empezando en ceros. Escribe en un papel los pesos después de cada corrección. ¿En qué época deja de equivocarse?

**Ejercicio 4 (medio). Flores separables.** Con las flores iris (`load_iris()` de `sklearn.datasets`), toma solo las especies setosa y versicolor (las primeras 100 flores) y dos medidas: largo del sépalo y largo del pétalo (columnas 0 y 2). Entrena tu `entrenar_perceptron` y dibuja las flores con la recta final. ¿Son linealmente separables? ¿Qué pasa si usas versicolor y virginica (flores 50 a 149)?

**Ejercicio 5 (medio). Tres entradas.** Inventa una decisión con 3 entradas (como el partido) y crea 8 ejemplos con la respuesta que tú darías. Entrena el perceptrón con ellos. ¿Consigue aprender tus decisiones? Si no, ¿por qué crees que no?

## 🏆 Reto opcional: XOR con tres neuronas

Una sola neurona no puede hacer XOR, pero **tres sí**. Observa que

$$\text{XOR}(x_1, x_2) = \text{Y}\big(\,\text{O}(x_1, x_2),\; \text{NAND}(x_1, x_2)\,\big)$$

1. Comprueba la fórmula con una tabla para las 4 entradas.
2. Usa tus pesos de O, NAND (ejercicio 2) e Y para armar una función `xor(x)` que use solo la función `neurona` tres veces.
3. Dibuja las **dos rectas** de la primera capa (O y NAND) sobre los 4 puntos. ¿Qué región del plano queda entre las dos rectas?

¡Acabas de construir una **red neuronal de dos capas**! En RN 2 haremos que una red así aprenda sus pesos sola.

## 🎬 Cierre

Hoy aprendiste que:
- Una **neurona artificial** suma sus entradas multiplicadas por **pesos**, le agrega un **sesgo** y aplica una **activación** (aquí, un escalón).
- Con dos entradas, un perceptrón es una **recta** que divide el plano: los pesos dan la inclinación y el sesgo la desplaza.
- La **regla del perceptrón** corrige los pesos solo cuando se equivoca: $w \leftarrow w + \eta \cdot \text{error} \cdot x$.
- Si los datos son **linealmente separables**, el perceptrón siempre aprende a separarlos; si no (como el **XOR**), nunca termina.
- La solución es conectar **varias neuronas en capas**.

**Para pensar:** en el reto armaste a mano una red de dos capas. Pero, ¿cómo podría una red **aprender** sus pesos sola, si la regla del perceptrón solo sabe corregir la última neurona? En la próxima lección construiremos una **red neuronal desde cero con numpy**, cambiaremos el escalón por una curva suave y la veremos resolver el XOR. 🧠